In [ ]:
# In[1]:

import json

# Define the task directory
task_directory = "rca100/cases/t081"

# Load the task.json file to understand the alert context
task_json_path = f"{task_directory}/task.json"
with open(task_json_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from task.json
alert_title = task_data.get("alert_title", None)
alert_window = task_data.get("alert_window", {})
alert_entity = task_data.get("alert_entity", None)
prompt_text = task_data.get("prompt_text", None)

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "checkout 错误次数告警" (checkout error count alert). It occurred between 2026-03-28 04:40:41.611202 and 2026-03-28 04:44:26 (UTC+8). The entity that triggered the alert is an APM operation named "checkout::/oteldemo.CheckoutService/PlaceOrder" with the entity ID "d219413245b68b297976412bbee076cf". The user is requesting root cause analysis for this alert.

The original code execution output of IPython Kernel is also provided below for reference:

('checkout 错误次数告警', {'start': '2026-03-28T04:40:41.611202+08:00', 'end': '2026-03-28T04:44:26+08:00'}, {'entity_id': 'd219413245b68b297976412bbee076cf', 'entity_name': 'checkout::/oteldemo.CheckoutService/PlaceOrder', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">checkout 错误次数告警<alert_event event_id="4e4e9b5bb21b453dadfec12cacd4db4b" trans_id="97cg9u3nbd77f2dp6p4nutom50" rule_id="08493be2-a181-4480-9895-2b2c5607122e" rule_name="checkout 错误次数告警" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[2]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[3]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[3]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[4]:

import pandas as pd

# Load the metrics.parquet file to analyze the metrics
metrics_path = f"{task_directory}/metrics.parquet"
metrics_data = pd.read_parquet(metrics_path)

# Display the available KPIs and their corresponding components
available_kpis = metrics_data[['entity_set', 'entity_name', 'metric']].drop_duplicates()

available_kpis

```
Out[4]:
```
The `metrics.parquet` file contains 278 unique combinations of KPIs and components. These include metrics for Kubernetes nodes (e.g., `node_ready_status`, `node_memory_usage_rate`, `node_disk_usage_rate`) and APM instances (e.g., `error_rate` for services like `frontend` and `frontend-proxy`). The data provides a wide range of metrics for analyzing system performance and detecting anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

entity_set                      entity_name                  metric
0          k8s.node                                        node_ready_status
1980       k8s.node                                   node_memory_usage_rate
3828       k8s.node                                     node_disk_usage_rate
4356       k8s.node                                   node_pod_running_count
6600       k8s.node                                    node_not_ready_status
...             ...                              ...                     ...
81514  apm.instance        frontend-6df7d557f5-nbvwh              error_rate
81563  apm.instance  frontend-proxy-67d64dc4f8-8c99b              error_rate
81614  apm.instance  frontend-proxy-67d64dc4f8-4ptg6              error_rate
81663  apm.instance        frontend-6df7d557f5-sgfcb              error_rate
81712  apm.instance        frontend-6df7d557f5-nz7g4              error_rate

[278 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import numpy as np

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Create a new column to classify by 'component-KPI'
metrics_data['component_kpi'] = metrics_data['entity_name'] + '-' + metrics_data['metric']

# Group by 'component-KPI' and calculate the P95 threshold for each group
p95_thresholds = metrics_data.groupby('component_kpi')['value'].quantile(0.95).reset_index()
p95_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

p95_thresholds

```
Out[5]:
```
The global P95 thresholds for each 'component-KPI' time series have been calculated across the entire dataset. There are 278 unique 'component-KPI' combinations, each with its corresponding P95 threshold. These thresholds represent the upper limit of normal behavior for each metric and component, which can be used to identify anomalies during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi  p95_threshold
0                    -apiserver_concurrent_connections       5.000000
1                  -apiserver_non_2xx_read_request_qps       0.000000
2                 -apiserver_non_2xx_write_request_qps       0.176826
3                       -apiserver_requests_per_second      26.917173
4                           -cluster_failed_node_count       0.000000
..                                                 ...            ...
273                                  shipping-workload    6345.250000
274  shipping::oteldemo.ShippingService/GetQuote-av...       0.002792
275  shipping::oteldemo.ShippingService/GetQuote-re...    5675.000000
276  shipping::oteldemo.ShippingService/ShipOrder-a...       0.000014
277  shipping::oteldemo.ShippingService/ShipOrder-r...    3271.000000

[278 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat("2026-03-28T04:40:41.611202+08:00").astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat("2026-03-28T04:44:26+08:00").astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the 'time' column to datetime in UTC+8
metrics_data['time'] = pd.to_datetime(metrics_data['time'] / 1_000_000, unit='s').dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')

# Filter the metrics data to include only the alert time window
filtered_metrics = metrics_data[(metrics_data['time'] >= alert_start) & (metrics_data['time'] <= alert_end)]

# Merge the filtered metrics with the P95 thresholds
anomalies = filtered_metrics.merge(p95_thresholds, on='component_kpi', how='left')

# Identify anomalies where the value exceeds the P95 threshold
anomalies = anomalies[anomalies['value'] > anomalies['p95_threshold']]

anomalies

```
Out[6]:
```
During the alert time window, 319 anomalies were identified where the metric values exceeded their global P95 thresholds. These anomalies span various components and KPIs, including Kubernetes node metrics (e.g., `node_disk_usage_rate`) and APM instance metrics (e.g., `error_rate` for `frontend` and `frontend-proxy`). These anomalies indicate potential performance issues or faults in the system during the alert period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain    entity_set                         entity_id                      entity_name  ...      value                                     metric_set_id service                               component_kpi p95_threshold
399  2026-03-28 04:41:11+08:00    k8s      k8s.node                                                                     ...  32.867365  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.656998
400  2026-03-28 04:41:41+08:00    k8s      k8s.node                                                                     ...  32.872790  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.656998
401  2026-03-28 04:42:11+08:00    k8s      k8s.node                                                                     ...  32.875019  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.656998
402  2026-03-28 04:42:41+08:00    k8s      k8s.node                                                                     ...  32.877613  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.656998
403  2026-03-28 04:43:11+08:00    k8s      k8s.node                                                                     ...  32.880061  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.656998
...                        ...    ...           ...                               ...                              ...  ...        ...                                               ...     ...                                         ...           ...
4312 2026-03-28 04:43:41+08:00    apm  apm.instance  fa548039d9a7b76db9ec70507e734faf  frontend-proxy-67d64dc4f8-8c99b  ...   0.191158            apm@metric_set@apm.metric.apm.instance    None  frontend-proxy-67d64dc4f8-8c99b-error_rate      0.186973
4313 2026-03-28 04:44:11+08:00    apm  apm.instance  fa548039d9a7b76db9ec70507e734faf  frontend-proxy-67d64dc4f8-8c99b  ...   0.192224            apm@metric_set@apm.metric.apm.instance    None  frontend-proxy-67d64dc4f8-8c99b-error_rate      0.186973
4316 2026-03-28 04:42:11+08:00    apm  apm.instance  eab531e9b188fe065edbebdbf839c609  frontend-proxy-67d64dc4f8-4ptg6  ...   0.206343            apm@metric_set@apm.metric.apm.instance    None  frontend-proxy-67d64dc4f8-4ptg6-error_rate      0.202119
4325 2026-03-28 04:43:11+08:00    apm  apm.instance  009ec66a350bc3e5210aea3dc6e6bdfd        frontend-6df7d557f5-sgfcb  ...   0.213872            apm@metric_set@apm.metric.apm.instance    None        frontend-6df7d557f5-sgfcb-error_rate      0.213498
4334 2026-03-28 04:44:11+08:00    apm  apm.instance  beecfe3febef4ed2425bac5626962ed9        frontend-6df7d557f5-nz7g4  ...   0.211320            apm@metric_set@apm.metric.apm.instance    None        frontend-6df7d557f5-nz7g4-error_rate      0.205939

[319 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Identify consecutive anomalies for each 'component-KPI' time series
# Sort anomalies by 'component_kpi' and 'time' to ensure proper grouping
anomalies = anomalies.sort_values(by=['component_kpi', 'time'])

# Add a helper column to identify consecutive anomalies
# A new group starts when the time difference between consecutive rows is greater than the sampling interval
anomalies['time_diff'] = anomalies.groupby('component_kpi')['time'].diff().dt.total_seconds()
sampling_interval = anomalies['time_diff'].median()  # Estimate the median sampling interval
anomalies['fault_group'] = (anomalies['time_diff'] > sampling_interval).cumsum()

# Filter out isolated noise spikes by keeping only groups with more than one anomaly
significant_faults = anomalies.groupby(['component_kpi', 'fault_group']).filter(lambda x: len(x) > 1)

significant_faults

```
Out[7]:
```
During the alert time window, 273 significant faults were identified across various 'component-KPI' time series. These faults represent consecutive anomalies where metric values consistently exceeded their global P95 thresholds, filtering out isolated noise spikes. The faults span multiple components, including Kubernetes deployments and APM operations, indicating sustained performance issues or faults in the system during the alert period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain      entity_set                         entity_id                                  entity_name  ... service                                      component_kpi p95_threshold time_diff fault_group
2285 2026-03-28 04:41:11+08:00    k8s  k8s.deployment                                                                                 ...    None                        -deployment_cpu_usage_total      0.243899       NaN           0
2418 2026-03-28 04:41:11+08:00    k8s  k8s.deployment                                                                                 ...    None                        -deployment_cpu_usage_total      0.243899       0.0           0
2286 2026-03-28 04:41:41+08:00    k8s  k8s.deployment                                                                                 ...    None                        -deployment_cpu_usage_total      0.243899      30.0           0
2419 2026-03-28 04:41:41+08:00    k8s  k8s.deployment                                                                                 ...    None                        -deployment_cpu_usage_total      0.243899       0.0           0
2287 2026-03-28 04:42:11+08:00    k8s  k8s.deployment                                                                                 ...    None                        -deployment_cpu_usage_total      0.243899      30.0           0
...                        ...    ...             ...                               ...                                          ...  ...     ...                                                ...           ...       ...         ...
3438 2026-03-28 04:43:41+08:00    apm    apm.instance  c73bdaaecdd2e6ba2c0ac60e95d1680f                               recommendation  ...    None                       recommendation-request_count    424.500000      30.0          17
2956 2026-03-28 04:42:41+08:00    apm   apm.operation  d4bd2837257aa9b4e96fa92507b51d78  shipping::oteldemo.ShippingService/GetQuote  ...    None  shipping::oteldemo.ShippingService/GetQuote-re...   5675.000000       NaN          17
2957 2026-03-28 04:43:11+08:00    apm   apm.operation  d4bd2837257aa9b4e96fa92507b51d78  shipping::oteldemo.ShippingService/GetQuote  ...    None  shipping::oteldemo.ShippingService/GetQuote-re...   5675.000000      30.0          17
2958 2026-03-28 04:43:41+08:00    apm   apm.operation  d4bd2837257aa9b4e96fa92507b51d78  shipping::oteldemo.ShippingService/GetQuote  ...    None  shipping::oteldemo.ShippingService/GetQuote-re...   5675.000000      30.0          17
2959 2026-03-28 04:44:11+08:00    apm   apm.operation  d4bd2837257aa9b4e96fa92507b51d78  shipping::oteldemo.ShippingService/GetQuote  ...    None  shipping::oteldemo.ShippingService/GetQuote-re...   5675.000000      30.0          17

[273 rows x 13 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

# Load the traces.parquet file
traces_path = f"{task_directory}/traces.parquet"
traces_data = pd.read_parquet(traces_path)

# Ensure relevant columns are numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Convert the alert time window to nanoseconds for comparison
alert_start_ns = int(alert_start.timestamp() * 1_000_000_000)
alert_end_ns = int(alert_end.timestamp() * 1_000_000_000)

# Filter traces related to the 'checkout::/oteldemo.CheckoutService/PlaceOrder' operation during the alert time window
filtered_traces = traces_data[
    (traces_data['spanName'] == 'checkout::/oteldemo.CheckoutService/PlaceOrder') &
    (traces_data['startTime'] >= alert_start_ns) &
    (traces_data['endTime'] <= alert_end_ns)
]

# Analyze the call chain to identify the most downstream faulty service
# Filter for spans with errors (statusCode != 0 or non-empty statusMessage)
filtered_traces['statusCode'] = pd.to_numeric(filtered_traces['statusCode'], errors='coerce')
faulty_traces = filtered_traces[
    (filtered_traces['statusCode'] != 0) | (filtered_traces['statusMessage'].notna())
]

# Sort by startTime to identify the most downstream faulty service
faulty_traces_sorted = faulty_traces.sort_values(by='startTime')

faulty_traces_sorted

```
Out[8]:
```
No faulty traces were found for the `checkout::/oteldemo.CheckoutService/PlaceOrder` operation during the alert time window. This indicates that there were no errors or anomalies directly associated with this operation in the distributed traces.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, events, __topic__, __tag__:__pack_id__, links, ip]
Index: []

[0 rows x 21 columns]```
```

In [ ]:
# In[9]:

# Adjust the code to handle already tz-aware datetime values
# Load the logs.parquet file
logs_path = f"{task_directory}/logs.parquet"
logs_data = pd.read_parquet(logs_path)

# Convert the '_time_' column to datetime in UTC+8 (handle already tz-aware values)
logs_data['_time_'] = pd.to_datetime(logs_data['_time_'], errors='coerce')
if logs_data['_time_'].dt.tz is None:
    logs_data['_time_'] = logs_data['_time_'].dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')
else:
    logs_data['_time_'] = logs_data['_time_'].dt.tz_convert('Asia/Shanghai')

# Filter logs related to the 'checkout' service or its dependencies during the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start) &
    (logs_data['_time_'] <= alert_end) &
    (logs_data['_container_name_'].str.contains('checkout', case=False, na=False))
]

# Search for error messages or anomalies in the log content
error_logs = filtered_logs[filtered_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[9]:
```
No error messages or anomalies were found in the logs related to the `checkout` service or its dependencies during the alert time window. This suggests that there were no logged issues directly associated with the `checkout` service during this period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, _image_name_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[10]:

# Calculate the magnitude of deviation for each significant fault
significant_faults['deviation'] = significant_faults['value'] - significant_faults['p95_threshold']

# Identify the component and KPI with the highest magnitude of deviation
max_deviation_fault = significant_faults.loc[significant_faults['deviation'].idxmax()]

max_deviation_fault

```
Out[10]:
```
The most significant deviation from the global P95 threshold occurred in the `deployment_memory_usage_total` KPI for a Kubernetes deployment. The metric value reached 3,498,516,480, exceeding the P95 threshold of 2,053,275,238.4 by 1,445,241,241.6. This indicates a substantial memory usage issue in the deployment during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

time                                     2026-03-28 04:42:11+08:00
domain                                                         k8s
entity_set                                          k8s.deployment
entity_id                                                         
entity_name                                                       
metric                               deployment_memory_usage_total
value                                                 3498516480.0
metric_set_id    k8s@metric_set@k8s.metric.high_level_metric_de...
service                                                       None
component_kpi                       -deployment_memory_usage_total
p95_threshold                                         2053275238.4
time_diff                                                      0.0
fault_group                                                      0
deviation                                             1445241241.6
Name: 1953, dtype: object```
```